# 03 modelos iniciales y submissions


### Modelo 4 — Incorporación del motivo del préstamo

Hasta acá trabajábamos con variables numéricas. En esta prueba agregamos Motivo_Prestamo mediante One-Hot Encoding y volvimos a calcular las probabilidades y la ganancia de la cartera. Queríamos comprobar si conocer el destino del préstamo ayudaba a distinguir solicitudes con distinto riesgo. La evaluación financiera nos sirvió para comparar esta opción con las anteriores, sin dar por hecho que agregar una columna siempre mejora el resultado.

In [ ]:
# hasta ahora veniamos ignorando el motivo del prestamo, y ya habiamos visto que determinados motivos tenian mas riesgo de default creo
# con one-hot encoding podemos representar las categorias como numeros para que el modelo las entienda


In [ ]:
# creamos otro X
X_motivo = train[
    [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso",
        "Motivo_Prestamo"
    ]
].copy()

In [ ]:
# usamos los X e y de train y val, no hacemos otro split
X_train_motivo = X_motivo.loc[X_train.index]
X_val_motivo = X_motivo.loc[X_val.index]


In [ ]:
# DEPENDENCIAS PARA ONE-HOT ENCODER
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [ ]:
columnas_categoricas = ["Motivo_Prestamo"]

In [ ]:
# creamos el preprocesador, sirve para decirle al modelo que transformacion aplicar a cada tipo de coluna antes de entrenar
preprocesador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            columnas_categoricas
        )
    ],
    remainder="passthrough"
)
# solo necesitamos transformar Motivo_Prestamo con One-Hot Encoding


In [ ]:
# CREAMOS MODELO
modelo_onehot = Pipeline([
    ("preprocesamiento", preprocesador),
    ("modelo", LogisticRegression(max_iter=1000))
])

In [ ]:
# ENTRENAMOS MODELO
modelo_onehot.fit(
    X_train_motivo,
    y_train
)

In [ ]:
# PREDICCIONES
predicciones_onehot = modelo_onehot.predict(
    X_val_motivo
)

probabilidades_onehot = modelo_onehot.predict_proba(
    X_val_motivo
)[:, 1]

In [ ]:
# metricas obtenidas
print(
    "Accuracy:",
    accuracy_score(y_val, predicciones_onehot)
)

print(
    confusion_matrix(y_val, predicciones_onehot)
)

print(
    classification_report(y_val, predicciones_onehot)
)

In [ ]:
# toma de train solo las filas q pertenecen a nuestro validation
# recuperamos los mismos prestamos con los que evaluamos el modelo, pero necesitamos las columnas q no estaban en Xvalidation, osea monto, tasa y resultado
cartera_val_onehot = train.loc[
    X_val.index,
    [
        "ID_Solicitud",
        "Monto_Solicitado",
        "Tasa_Interes",
        "Estado_Final"
    ]
].copy()

In [ ]:
# agregamos a cada prestamo la probabilidad estimada por el mdodelo de q ese cliente pague
cartera_val_onehot["Prob_Pago"] = probabilidades_onehot
#calculamos probabilidad complementaria a anterior
cartera_val_onehot["Prob_Default"] = 1 - cartera_val_onehot["Prob_Pago"]

In [ ]:
# EV, convertimos la prediccion del modelo en el valor esperado de cada prestamo
cartera_val_onehot["EV"] = (
    cartera_val_onehot["Prob_Pago"]
    * cartera_val_onehot["Monto_Solicitado"]
    * cartera_val_onehot["Tasa_Interes"]
    -
    cartera_val_onehot["Prob_Default"]
    * cartera_val_onehot["Monto_Solicitado"]
)

In [ ]:
# para ordenar prestamos desde mejor financieramente a menos atractivo, osea segun el EV
cartera_val_onehot = cartera_val_onehot.sort_values(
    "EV",
    ascending=False
)

In [ ]:
# aplicamos el limite de presupuesto
presupuesto = 2_000_000
capital_usado_onehot = 0

cartera_val_onehot["Aprobar"] = 0

for indice, fila in cartera_val_onehot.iterrows():

    if fila["EV"] <= 0:
        continue

    if capital_usado_onehot + fila["Monto_Solicitado"] <= presupuesto:
        cartera_val_onehot.loc[indice, "Aprobar"] = 1
        capital_usado_onehot += fila["Monto_Solicitado"]

In [ ]:
print("Capital utilizado:", capital_usado_onehot)
print("Préstamos aprobados:", cartera_val_onehot["Aprobar"].sum())

In [ ]:
# CALCULO DE CTO HABRIAMOS GANADO EN VALIDATION
def ganancia_real_onehot(fila):

    if fila["Aprobar"] == 0:
        return 0

    if fila["Estado_Final"] == 1:
        return fila["Monto_Solicitado"] * fila["Tasa_Interes"]

    return -fila["Monto_Solicitado"]

In [ ]:
cartera_val_onehot["Ganancia_Real"] = cartera_val_onehot.apply(
    ganancia_real_onehot,
    axis=1
)

ganancia_onehot = cartera_val_onehot["Ganancia_Real"].sum()

print("Ganancia real simulada:", ganancia_onehot)
# mejoro con respecto al scaler pero sigue siendo pero q la regresion logistica original
# La ganancia simulada fue de aprox $260.070, mientras que la regresión logística original obtuvo $347.031.

### Comparación financiera de los modelos iniciales

Comparamos las decisiones de las regresiones iniciales usando la ganancia o pérdida que establece la consigna. Esto nos mostró por qué un modelo con mejor accuracy no necesariamente deja más dinero: prestar a un cliente que incumple cuesta el capital completo, mientras que uno que paga deja solo intereses.

In [ ]:
evaluacion = train.loc[X_val.index, [
    "Monto_Solicitado",
    "Tasa_Interes"
]].copy()

evaluacion["Estado_Real"] = y_val
evaluacion["Pred_Original"] = predicciones
evaluacion["Pred_Balanceada"] = predicciones_balanceadas

evaluacion.head()

In [ ]:
# calcular ganancias
def calcular_ganancia(fila, columna_prediccion):
    if fila[columna_prediccion] == 0:
        return 0

    if fila["Estado_Real"] == 1:
        return fila["Monto_Solicitado"] * fila["Tasa_Interes"]

    return -fila["Monto_Solicitado"]

In [ ]:
#calculo ganancia baseline 1
evaluacion["Ganancia_Original"] = evaluacion.apply(
    lambda fila: calcular_ganancia(fila, "Pred_Original"),
    axis=1
)
#calculo baseline 2
evaluacion["Ganancia_Balanceada"] = evaluacion.apply(
    lambda fila: calcular_ganancia(fila, "Pred_Balanceada"),
    axis=1
)

In [ ]:
print(
    "Ganancia baseline original:",
    evaluacion["Ganancia_Original"].sum()
)

print(
    "Ganancia baseline balanceada:",
    evaluacion["Ganancia_Balanceada"].sum()
)
 #vemos que ambas baselines ganan plata, pero a que costo...

In [ ]:
capital_original = evaluacion.loc[
    evaluacion["Pred_Original"] == 1,
    "Monto_Solicitado"
].sum()

capital_balanceada = evaluacion.loc[
    evaluacion["Pred_Balanceada"] == 1,
    "Monto_Solicitado"
].sum()

print("Capital prestado original:", capital_original)
print("Capital prestado balanceada:", capital_balanceada)
# puntaje negativo en el kaggle, se recontra pasaron de 2millones

### Primera selección de préstamos por valor esperado

En lugar de aprobar directamente con `.predict()`, calculamos cuánto esperamos ganar o perder en cada solicitud a partir de su probabilidad de pago. Ordenamos los préstamos y aprobamos los que entraban en el límite de USD 2 millones. Luego medimos el resultado con las etiquetas conocidas de validación. Esta fue la base de las estrategias financieras que probamos después.

In [ ]:
cartera_val = train.loc[X_val.index, [
    "ID_Solicitud",
    "Monto_Solicitado",
    "Tasa_Interes",
    "Estado_Final"
]].copy()

cartera_val["Prob_Pago"] = probabilidades

cartera_val.head()
# tabla para ir trabajando

In [ ]:
# calcular esperanza matematica, la regla economica del tp que es EV = P(paga) (monto * tasa) + P(default) (-Monto)
# pdefault) = 1 - P(paga)

cartera_val["Prob_Default"] = 1 - cartera_val["Prob_Pago"]

cartera_val["EV"] = (
    cartera_val["Prob_Pago"]
    * cartera_val["Monto_Solicitado"]
    * cartera_val["Tasa_Interes"]
    -
    cartera_val["Prob_Default"]
    * cartera_val["Monto_Solicitado"]
)

cartera_val[
    ["Monto_Solicitado", "Tasa_Interes", "Prob_Pago", "EV"]
].head()
# EV > 0 significa que segun el model prestar tiene ganancia esperada positiva
# EV < 0 significa que seugn el modelo el riesgo de prestar no compensa los intereses

In [ ]:
cartera_val = cartera_val.sort_values("EV", ascending=False)
# pone arriba los prestamos que el modelo considera mas rentables

In [ ]:
presupuesto = 2_000_000
capital_usado = 0

cartera_val["Aprobar"] = 0

for indice, fila in cartera_val.iterrows():

    if fila["EV"] <= 0:
        continue

    if capital_usado + fila["Monto_Solicitado"] <= presupuesto:
        cartera_val.loc[indice, "Aprobar"] = 1
        capital_usado += fila["Monto_Solicitado"]

# aca asignamos el presupuesto
# empezamos por el mejor EV, si es positivo y esta dentro del presupuesto: se aprueba y se resta del presupuesto
# apsamos al siguiente prestamo

In [ ]:
print("Capital utilizado:", capital_usado)
print("Préstamos aprobados:", cartera_val["Aprobar"].sum())
# para ver cuanto dinero prestamos y cuantos prestamos aprobamos

In [ ]:
# como conocemos Estado_Final podemos calcular cuanto habria ganado realmente nustra estrategia
# CODIGO PARA PROBAR ESTRATEGIA, NO ENTREGAR
def ganancia_real(fila):
    if fila["Aprobar"] == 0:
        return 0

    if fila["Estado_Final"] == 1:
        return fila["Monto_Solicitado"] * fila["Tasa_Interes"]

    return -fila["Monto_Solicitado"]

cartera_val["Ganancia_Real"] = cartera_val.apply(
    ganancia_real,
    axis=1
)

ganancia_total = cartera_val["Ganancia_Real"].sum()

print("Ganancia real simulada:", ganancia_total)

### Submission V1 — Primera entrega a Kaggle

Entrenamos la regresión logística inicial con los datos disponibles y armamos el archivo de decisiones. Revisamos IDs, presupuesto y formato. La primera entrega no dio un resultado favorable, como anotamos en el propio código, así que continuamos trabajando sobre el historial bancario y la selección financiera.

In [ ]:
# usamos la regresion logistica original ya q fue el modelo que obtuvo la mayor ganancia simulada sobre validation
# para la submission se entrena utilizando todos los datos de entrenamiento dispoibles

from sklearn.linear_model import LogisticRegression

modelo_final = LogisticRegression(max_iter=1000)

modelo_final.fit(X, y)

In [ ]:
# cargamos las solicitudes que debemos evaluar
test = pd.read_csv("solicitudes_test.csv")
test.head()


In [ ]:
test.info()

In [ ]:
# creamos la misma feature que en train, el ratio monto ingreso
test["Ratio_Monto_Ingreso"] = (
    test["Monto_Solicitado"] / test["Ingreso_Mensual"]
)

In [ ]:
# aca seleccionamos las mismas features que en train, armamos  X
X_test = test[
    [
        "Monto_Solicitado",
        "Tasa_Interes",
        "Ingreso_Mensual",
        "Ratio_Monto_Ingreso"
    ]
]

In [ ]:
# predecimos las probabilidades
prob_test = modelo_final.predict_proba(X_test)[:, 1]

# aca obtenemos P(paga)

In [ ]:
# para ver algunas probabilidades de pago
prob_test[:20]

In [ ]:
# armamos la cartera de kaggle
cartera_test = test[
    [
        "ID_Solicitud",
        "Monto_Solicitado",
        "Tasa_Interes"
    ]
].copy()

cartera_test["Prob_Pago"] = prob_test
cartera_test["Prob_Default"] = 1 - cartera_test["Prob_Pago"]

In [ ]:
# calculamos esperanza EV
cartera_test["EV"] = (
    cartera_test["Prob_Pago"]
    * cartera_test["Monto_Solicitado"]
    * cartera_test["Tasa_Interes"]
    -
    cartera_test["Prob_Default"]
    * cartera_test["Monto_Solicitado"]
)

In [ ]:
# establecemos los limites de la consigna
cartera_test = cartera_test.sort_values("EV", ascending=False)

presupuesto = 2_000_000
capital_usado = 0

cartera_test["Aprobar"] = 0

for indice, fila in cartera_test.iterrows():

    if fila["EV"] <= 0:
        continue

    if capital_usado + fila["Monto_Solicitado"] <= presupuesto:
        cartera_test.loc[indice, "Aprobar"] = 1
        capital_usado += fila["Monto_Solicitado"]

In [ ]:
# verificacion
print("Capital utilizado:", capital_usado)
print("Préstamos aprobados:", cartera_test["Aprobar"].sum())

In [ ]:
test.shape

In [ ]:
test.columns

In [ ]:
test.head()

In [ ]:
# generar csv submission
submission_v1 = test[["ID_Solicitud"]].copy()

submission_v1 = submission_v1.merge(
    cartera_test[["ID_Solicitud", "Aprobar"]],
    on="ID_Solicitud",
    how="left"
)

submission_v1 = submission_v1.rename(
    columns={
        "Aprobar": "Decision_Aprobacion"
    }
)

In [ ]:
# para verificar
print("Dimensiones:", submission_v1.shape)

print(submission_v1["Decision_Aprobacion"].value_counts())

print("Valores nulos:", submission_v1.isnull().sum())

submission_v1.head()

In [ ]:
# para revisar que no nos hayamos pasado de dinero prestado
control_submission = test.merge(
    submission_v1,
    on="ID_Solicitud"
)

capital_submission = control_submission.loc[
    control_submission["Decision_Aprobacion"] == 1,
    "Monto_Solicitado"
].sum()

print(
    "Capital total aprobado:",
    capital_submission
)

In [ ]:
# exportar
submission_v1.to_csv(
    "submission_v1.csv",
    index=False
)
pd.read_csv("submission_v1.csv").head()

In [ ]:

print("Archivo disponible para descargar:", "submission_v1.csv")
# RESULTADO: DIO NEGATIVO.......

### Submission V2 — Otra selección de cartera

Volvimos a usar las probabilidades del modelo inicial para probar otra forma de decidir qué préstamos aprobar. Comparamos capital asignado y ganancia en validación antes de generar el archivo. Esta versión forma parte de las pruebas iniciales; la selección definitiva se hizo con experimentos posteriores.

In [ ]:
# partimos de la misma validacion y mismas prob qeu el modelo original
cartera_val_v2 = train.loc[
    X_val.index,
    [
        "ID_Solicitud",
        "Monto_Solicitado",
        "Tasa_Interes",
        "Estado_Final"
    ]
].copy()

cartera_val_v2["Prob_Pago"] = probabilidades
cartera_val_v2["Prob_Default"] = 1 - cartera_val_v2["Prob_Pago"]

In [ ]:
# calculamos el EV de siempre
cartera_val_v2["EV"] = (
    cartera_val_v2["Prob_Pago"]
    * cartera_val_v2["Monto_Solicitado"]
    * cartera_val_v2["Tasa_Interes"]
    -
    cartera_val_v2["Prob_Default"]
    * cartera_val_v2["Monto_Solicitado"]
)

In [ ]:
# EV que dice cuanto valor esperado genera cada dolar prestado
cartera_val_v2["EV_por_dolar"] = (
    cartera_val_v2["EV"]
    / cartera_val_v2["Monto_Solicitado"]
)

In [ ]:
#ordenamos por esa metrica
cartera_val_v2 = cartera_val_v2.sort_values(
    "EV_por_dolar",
    ascending=False
)

In [ ]:
# presupuesto
presupuesto = 2_000_000
capital_usado_v2 = 0

cartera_val_v2["Aprobar"] = 0

for indice, fila in cartera_val_v2.iterrows():

    if fila["EV"] <= 0:
        continue

    if capital_usado_v2 + fila["Monto_Solicitado"] <= presupuesto:
        cartera_val_v2.loc[indice, "Aprobar"] = 1
        capital_usado_v2 += fila["Monto_Solicitado"]

In [ ]:
# calculo ganancia real
def ganancia_real_v2(fila):

    if fila["Aprobar"] == 0:
        return 0

    if fila["Estado_Final"] == 1:
        return fila["Monto_Solicitado"] * fila["Tasa_Interes"]

    return -fila["Monto_Solicitado"]

In [ ]:
cartera_val_v2["Ganancia_Real"] = cartera_val_v2.apply(
    ganancia_real_v2,
    axis=1
)

ganancia_v2 = cartera_val_v2["Ganancia_Real"].sum()

print("Capital utilizado:", capital_usado_v2)
print("Préstamos aprobados:", cartera_val_v2["Aprobar"].sum())
print("Ganancia real simulada:", ganancia_v2)

#CONCLUSION: no conviene usarla, aumento la cantidad de prestamos aceptados pero redujo la ganancia de 347 a 234